# BCI Decoder Training — EEGNet & LSTM

Train EEGNet and LSTM decoders on preprocessed EEG data (HDF5) with GPU.

**Prerequisites:** Upload `S01_preprocessed.h5` and `S05_preprocessed.h5` to Google Drive under `My Drive/bci/preprocessed/`.

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo
!git clone https://github.com/Qwant07/rl-eeg-cursor.git /content/rl-eeg-cursor
%cd /content/rl-eeg-cursor

In [ ]:
# Install dependencies (torch + h5py already in Colab)
!pip install -q mne scikit-learn

In [ ]:
# Symlink preprocessed data from Drive
import os

# ===== EDIT THIS PATH if your Drive folder is different =====
DRIVE_DATA = '/content/drive/MyDrive/bci/preprocessed'

os.makedirs('preprocessed', exist_ok=True)
for fname in ['S01_preprocessed.h5', 'S05_preprocessed.h5']:
    src = os.path.join(DRIVE_DATA, fname)
    dst = os.path.join('preprocessed', fname)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'Linked {fname}')
    elif os.path.exists(dst):
        print(f'{fname} already linked')
    else:
        print(f'WARNING: {src} not found — upload it to Drive first!')

In [ ]:
# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Train EEGNet — S01

In [ ]:
!python -m src.decoders.train \
    --subject S01 \
    --model eegnet \
    --device cuda \
    --epochs 200 \
    --batch_size 128 \
    --lr 1e-3 \
    --weight_decay 1e-4 \
    --patience 20 \
    --num_workers 2 \
    --out_dir results

## 3. Train LSTM — S01

In [ ]:
!python -m src.decoders.train \
    --subject S01 \
    --model lstm \
    --device cuda \
    --epochs 200 \
    --batch_size 128 \
    --lr 1e-3 \
    --weight_decay 1e-4 \
    --patience 20 \
    --num_workers 2 \
    --out_dir results

## 4. Train EEGNet — S05

In [ ]:
!python -m src.decoders.train \
    --subject S05 \
    --model eegnet \
    --device cuda \
    --epochs 200 \
    --batch_size 128 \
    --lr 1e-3 \
    --weight_decay 1e-4 \
    --patience 20 \
    --num_workers 2 \
    --out_dir results

## 5. Train LSTM — S05

In [ ]:
!python -m src.decoders.train \
    --subject S05 \
    --model lstm \
    --device cuda \
    --epochs 200 \
    --batch_size 128 \
    --lr 1e-3 \
    --weight_decay 1e-4 \
    --patience 20 \
    --num_workers 2 \
    --out_dir results

## 6. LDA Baseline

In [ ]:
import json
import numpy as np
import h5py
from src.baselines.lda_decoder import BandPowerLDA

def load_sessions(h5_path, subject, sessions, max_samples=0):
    """Load X, y from HDF5 for given sessions."""
    X_parts, y_parts = [], []
    total = 0
    with h5py.File(h5_path, 'r') as f:
        subj = f[subject]
        for s in sessions:
            sk = f'session_{s:02d}'
            if sk not in subj:
                continue
            for rt in sorted(subj[sk]):
                for rk in sorted(subj[sk][rt]):
                    X_parts.append(subj[sk][rt][rk]['X'][:])
                    y_parts.append(subj[sk][rt][rk]['y'][:])
                    total += len(X_parts[-1])
                    if max_samples > 0 and total >= max_samples:
                        break
                if max_samples > 0 and total >= max_samples:
                    break
            if max_samples > 0 and total >= max_samples:
                break
    X = np.concatenate(X_parts)
    y = np.concatenate(y_parts)
    if max_samples > 0:
        X, y = X[:max_samples], y[:max_samples]
    return X, y

for subject in ['S01', 'S05']:
    h5_path = f'preprocessed/{subject}_preprocessed.h5'
    print(f'\n=== LDA Baseline: {subject} ===')
    X_train, y_train = load_sessions(h5_path, subject, [1, 2])
    X_val, y_val = load_sessions(h5_path, subject, [3])
    print(f'Train: {X_train.shape[0]} epochs, Val: {X_val.shape[0]} epochs')

    lda = BandPowerLDA(fs=1000.0)
    lda.fit(X_train, y_train)
    scores = lda.score(X_val, y_val)
    print(f"  R²  vx={scores['r2_vx']:.4f}  vy={scores['r2_vy']:.4f}  mean={scores['r2_mean']:.4f}")
    print(f"  NMSE vx={scores['nmse_vx']:.4f} vy={scores['nmse_vy']:.4f} mean={scores['nmse_mean']:.4f}")

    # Save
    from pathlib import Path
    out = Path('results') / subject / 'lda'
    out.mkdir(parents=True, exist_ok=True)
    with open(out / 'best_metrics.json', 'w') as f:
        json.dump(scores, f, indent=2)
    print(f'  Saved → {out / "best_metrics.json"}')

## 7. Compare All Models

In [ ]:
import json
from pathlib import Path

print(f'{"Subject":<10} {"Model":<10} {"R² vx":>8} {"R² vy":>8} {"R² mean":>8} {"NMSE mean":>10}')
print('-' * 60)

for subject in ['S01', 'S05']:
    for model in ['lda', 'eegnet', 'lstm']:
        path = Path('results') / subject / model / 'best_metrics.json'
        if not path.exists():
            print(f'{subject:<10} {model:<10} {"— not found —"}')
            continue
        with open(path) as f:
            m = json.load(f)
        print(f'{subject:<10} {model:<10} {m["r2_vx"]:>8.4f} {m["r2_vy"]:>8.4f} {m["r2_mean"]:>8.4f} {m["nmse_mean"]:>10.4f}')

## 8. Plot Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, subject in enumerate(['S01', 'S05']):
    for model, color in [('eegnet', 'tab:blue'), ('lstm', 'tab:orange')]:
        path = Path('results') / subject / model / 'history.json'
        if not path.exists():
            continue
        with open(path) as f:
            h = json.load(f)
        epochs = range(1, len(h['train_loss']) + 1)

        # Loss
        axes[0, col].plot(epochs, h['train_loss'], '--', color=color, alpha=0.5, label=f'{model} train')
        axes[0, col].plot(epochs, h['val_loss'], '-', color=color, label=f'{model} val')
        axes[0, col].set_title(f'{subject} — Loss')
        axes[0, col].set_xlabel('Epoch')
        axes[0, col].set_ylabel('MSE Loss')
        axes[0, col].legend()

        # R²
        axes[1, col].plot(epochs, h['val_r2_mean'], '-', color=color, label=f'{model}')
        axes[1, col].set_title(f'{subject} — Validation R²')
        axes[1, col].set_xlabel('Epoch')
        axes[1, col].set_ylabel('R² (mean)')
        axes[1, col].legend()

plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150)
plt.show()
print('Saved → results/training_curves.png')

## 9. Copy Results Back to Drive

In [ ]:
import shutil
dst = '/content/drive/MyDrive/bci/results'
if os.path.exists('results'):
    shutil.copytree('results', dst, dirs_exist_ok=True)
    print(f'Results copied to {dst}')
else:
    print('No results directory found')